<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/Lab_1_Lektion_5_Intelligenta_Algoritmer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Städrobot - Lektion 5: Intelligenta algoritmer

I den här lektionen ska vi göra vår städrobot **riktigt smart**!

Vi lär oss:
- ✅ Vad en **algoritm** är
- ✅ Skillnaden mellan slumpmässig, regelbaserad och **intelligent** rörelse
- ✅ Hur **BFS (Bredden-Först-Sökning)** fungerar
- ✅ Hur man hittar den **optimala vägen** från punkt A till B

**Bygger vidare på:** Lektion 4 (Regelbaserad AI)

⏱️ **Tid:** Ca 60 minuter

# 🔧 Setup - Importera verktyg

Kör den här cellen **först** – den laddar in alla verktyg vi behöver.

In [ ]:
import ipywidgets as widgets           # Verktyg för att skapa knappar och gränssnitt
from IPython.display import display, clear_output  # Verktyg för att visa och rensa utskrifter
import random                           # Slumptalsgenerator
import math                             # Matematikfunktioner
from collections import deque           # Effektiv kö för BFS

print("✅ Verktyg laddade!")

# 🌍 Del 0: Grundkod - Världen och roboten

Den här cellen innehåller all grundkod från tidigare lektioner.
**Kör den en gång** – sedan kan vi använda allt nedan!

*(Du behöver inte läsa allt, men titta gärna på kommentarerna)*

In [ ]:
# =============================================================
# GRUNDKOD - Världen, roboten och hjälpfunktioner
# =============================================================

# --- Celltyper (vad en ruta i världen kan vara) ---
class CellType:
  EMPTY   = 0  # Tom cell
  WALL    = 1  # Vägg
  DIRT    = 2  # Smuts
  HOME    = 3  # Robotens hem (startposition)
  UNKNOWN = 4  # Okänd cell

WALL = CellType.WALL  # Kortare namn för vägg

# --- Omvandla celltyp till tecken ---
def cell_to_string(cell):
  if cell == CellType.EMPTY: return "."
  if cell == CellType.WALL:  return "#"
  if cell == CellType.DIRT:  return "░"
  if cell == CellType.HOME:  return "H"
  return "?"

# --- Skapa en värld ---
def generate_world(grid_size, wall_density=0.1, dirt_density=0.1):
  area = (grid_size - 2) ** 2
  grid = [[CellType.EMPTY] * grid_size for _ in range(grid_size)]
  # Kantvägg
  for row in range(grid_size):
    grid[row][0] = grid[row][grid_size-1] = CellType.WALL
  for col in range(grid_size):
    grid[0][col] = grid[grid_size-1][col] = CellType.WALL
  # Inre väggar
  for _ in range(math.ceil(area * wall_density)):
    for _ in range(1000):
      r, c = random.randint(1, grid_size-2), random.randint(1, grid_size-2)
      if grid[r][c] == CellType.EMPTY:
        grid[r][c] = CellType.WALL
        break
  # Smuts
  for _ in range(math.ceil(area * dirt_density)):
    for _ in range(1000):
      r, c = random.randint(1, grid_size-2), random.randint(1, grid_size-2)
      if grid[r][c] == CellType.EMPTY:
        grid[r][c] = CellType.DIRT
        break
  grid[1][1] = CellType.HOME
  return grid

# --- Skriv ut världen ---
def print_grid(grid, robot_x=None, robot_y=None, path=None, visited=None, margin="  "):
  """Skriv ut rutnätet med valfri robot, stig och utforskade celler"""
  path_set    = set(path)    if path    else set()
  visited_set = set(visited) if visited else set()
  for y in range(len(grid)):
    row_str = margin
    for x in range(len(grid[y])):
      if robot_x == x and robot_y == y:
        row_str += "R "  # Robot
      elif (x, y) in path_set:
        row_str += "* "  # Del av vägen
      elif (x, y) in visited_set:
        row_str += "~ "  # Utforskad
      else:
        row_str += cell_to_string(grid[y][x]) + " "
    print(row_str)
  print()

# --- Agentåtgärder ---
AGENT_ACTION_NOOP        = 0
AGENT_ACTION_MOVE_LEFT   = 1
AGENT_ACTION_MOVE_DOWN   = 2
AGENT_ACTION_MOVE_RIGHT  = 3
AGENT_ACTION_MOVE_UP     = 4
AGENT_ACTION_SUCK_DIRT   = 5
AGENT_MOVE_ACTIONS = [AGENT_ACTION_MOVE_UP, AGENT_ACTION_MOVE_DOWN,
                      AGENT_ACTION_MOVE_LEFT, AGENT_ACTION_MOVE_RIGHT]

def action_to_string(action):
  names = {0:"NOOP", 1:"VÄNSTER", 2:"NER", 3:"HÖGER", 4:"UPP", 5:"DAMMSUG"}
  return names.get(action, f"?{action}")

# --- Hitta grannar (celler vi kan röra oss till) ---
def get_neighbors(pos, world):
  """Returnera alla positioner vi kan röra oss till från pos (ej väggar)"""
  x, y = pos
  neighbors = []
  for dx, dy in [(1,0), (-1,0), (0,1), (0,-1)]:  # höger, vänster, ner, upp
    nx, ny = x + dx, y + dy
    if 0 <= nx < len(world[0]) and 0 <= ny < len(world):
      if world[ny][nx] != CellType.WALL:
        neighbors.append((nx, ny))
  return neighbors

print("✅ Grundkod redo!")

---
# 🧠 Del 1: Introduktion - Vad är en algoritm?

## Berättelse
Vi har lärt oss att bygga en regelbaserad robot (Lektion 4).
Den följer enkla regler:
- "Om det finns smuts → dammsug"
- "Annars → rör dig höger"

**Men vad händer om roboten kör fast i ett hörn? 🤔**

---

## 📚 Tre sorters intelligens:

| Typ | Hur den fungerar | Problem |
|-----|------------------|---------|
| 🎲 **Slumpmässig** | Väljer riktning slumpmässigt | Mycket långsam |
| 📋 **Regelbaserad** | Följer fasta regler (Lektion 4) | Kan fastna i loopar |
| 🧠 **Intelligent (BFS)** | Beräknar optimal väg | Lite mer kod |

---

## 💡 Vad är en algoritm?

En **algoritm** är en **steg-för-steg-instruktion** för att lösa ett problem.

**Exempel:** Recept på pannkakor
```
Steg 1: Blanda mjöl, ägg och mjölk
Steg 2: Värm stekpannan
Steg 3: Häll i smet
Steg 4: Stek tills bubblig
Steg 5: Vänd och stek klart
```

En algoritm i datorer är **exakt samma sak** – fast för att lösa problem som:
"Hur tar jag mig från A till B på kortaste vägen?"

---
# 🗺️ Del 2: Problemet - Hur tar sig roboten från A till B?

**Scenario:**
- Robot på position **(1, 1)**
- Smuts på position **(5, 3)**
- Hur ska roboten ta sig dit?

Kör cellen nedan för att se världen:

In [ ]:
# Skapa en enkel värld för att illustrera problemet
# ---------------------------------------------------
random.seed(42)  # Fast slumpfrö = samma värld varje gång

# Skapa en 8x8 värld med lite väggar
world = generate_world(8, wall_density=0.05, dirt_density=0.0)

# Placera smuts manuellt på en specifik position
start_pos  = (1, 1)  # Robotens startposition
goal_pos   = (5, 3)  # Smuts/mål-position
world[goal_pos[1]][goal_pos[0]] = CellType.DIRT  # Placera smuts

print("🗺️ Världen:")
print("  R = Robot (startposition)")
print("  ░ = Smuts (mål)")
print("  # = Vägg")
print("  . = Tom")
print()
print_grid(world, robot_x=start_pos[0], robot_y=start_pos[1])

print(f"Roboten börjar på: {start_pos}")
print(f"Målet (smuts) är på: {goal_pos}")

### 🤔 Hur löser vi det?

```
1. Slumpmässig:   Gå åt ett slumpmässigt håll varje steg
                  ❌ Kan ta HUNDRATALS steg

2. Regelbaserad:  Gå höger om möjligt, annars ner, annars vänster...
                  ❌ Kan fastna i loopar/hörn

3. BFS-sökning:   Beräkna kortaste vägen direkt!
                  ✅ Garanterat optimal väg
```

---
# 🔍 Del 3: Vad är en sökalgorithm?

En sökalgorithm är en algoritm som **letar efter en väg** i ett utrymme.

**Grundidén:**
```
Start: Robot på (1,1)
Mål: Hitta vägen till (5,3)

Algoritmen frågar: "Vilka vägar kan jag ta från start?"
Sedan: "Vilka vägar kan jag ta från DESSA positioner?"
Tills: "Jag hittade målet!"
```

---

## Steg-för-steg exempel:

```
Steg 1: Kolla grannar till start (1,1)
  (1,1) → kan gå till (2,1), (1,2)

Steg 2: Kolla grannar till de positionerna
  (2,1) → kan gå till (3,1), (2,2)
  (1,2) → kan gå till (2,2), (1,3)

Steg 3: Fortsätt tills du hittar målet (5,3)
```

**Viktigt:** Vi håller koll på var vi har **redan varit** – annars loopar vi!

---

**📝 Uppgift 3.1:** Titta på koden nedan. Vad tror du `get_neighbors` gör?

In [ ]:
# get_neighbors - hitta alla grannar till en position
# -------------------------------------------------------
# Denna funktion är redan definierad i grundkoden ovan.
# Låt oss testa den!

test_pos = (3, 3)  # Testa från mitten av världen
grannar  = get_neighbors(test_pos, world)

print(f"Från position {test_pos} kan roboten gå till:")
for granne in grannar:
  print(f"  → {granne}")

print()
print("Världen runt (3,3):")
print_grid(world, robot_x=test_pos[0], robot_y=test_pos[1])

---
# 🌊 Del 4: BFS - Bredden-Först-Sökning

**BFS (Breadth-First Search)** är en algoritm som:
- Hittar den **kortaste vägen** (garanterat!)
- Utforskar alla grannar på samma **nivå** innan den går djupare
- Fungerar som **ringar på vattnet** – sprider sig utåt från start

---

## Pseudokod (enkel version):

```
BFS(start, mål):
  1. Lägg start i en KÖ
  2. Medan kön inte är tom:
     a. Ta ut första positionen från kön ("current")
     b. Är vi framme? → Returnera vägen!
     c. Markera current som besökt
     d. Lägg till alla GRANNAR i kön (om inte besökta)
  3. Ingen väg hittades
```

**🔑 Nyckelord:**
- **KÖ (queue):** Första in, första ut (som en kö på affären)
- **Besökt (visited):** Positioner vi redan kollat
- **Förälder (parent):** Varifrån kom vi till denna position?

---

## Visuellt:

```
Nivå 0:  Start (1,1)
Nivå 1:  (2,1) (1,2)          ← grannar till start
Nivå 2:  (3,1) (2,2) (1,3)   ← grannar till nivå 1
Nivå 3:  (4,1) (3,2) ...      ← grannar till nivå 2
...
Tills vi hittar (5,3)!
```

In [ ]:
# BFS - Bredden-Först-Sökning
# ============================
# Hittar den kortaste vägen från start till mål.

def bfs_sök_väg(start, mål, world):
  """
  Hitta kortaste vägen från start till mål med BFS.
  
  start, mål: tupler (x, y)
  world: 2D-lista med celltyper
  Returnerar: lista med positioner [(x1,y1), (x2,y2), ...] eller [] om ingen väg
  """

  # 1. Sätt upp kön med startpositionen
  kö      = deque([start])     # Kön av positioner att utforska
  besökt  = {start}            # Mängd av besökta positioner
  förälder = {start: None}     # Varifrån kom vi till varje position?

  # 2. Utforska tills kön är tom
  while kö:
    nuvarande = kö.popleft()   # Ta FÖRSTA positionen från kön (FIFO)

    # 3. Hittade vi målet?
    if nuvarande == mål:
      # Rekonstruera vägen bakifrån!
      väg = []
      pos = nuvarande
      while pos is not None:
        väg.append(pos)
        pos = förälder[pos]   # Gå tillbaka längs föräldralänkarna
      return väg[::-1]        # Vänd på listan (vi byggde den bakifrån)

    # 4. Utforska grannar
    for granne in get_neighbors(nuvarande, world):
      if granne not in besökt:
        besökt.add(granne)            # Markera som besökt
        förälder[granne] = nuvarande  # Kom ihåg varifrån vi kom
        kö.append(granne)             # Lägg till i kön

  return []  # Ingen väg hittades


print("✅ BFS-funktionen definierad!")

### 🧪 Testa BFS!

**Uppgift 4.1:** Kör cellen nedan för att se BFS i action.

Frågor att fundera på:
- Hur lång är vägen?
- Verkar det vara den kortaste möjliga vägen?

In [ ]:
# Testa BFS - hitta väg från start till mål
# ------------------------------------------

väg = bfs_sök_väg(start_pos, goal_pos, world)

if väg:
  print(f"✅ Väg hittad! ({len(väg)} positioner, {len(väg)-1} steg)")
  print(f"Vägen: {' → '.join(str(p) for p in väg)}")
  print()
  print("🗺️ Vägen markerad med * på kartan:")
  print_grid(world, robot_x=start_pos[0], robot_y=start_pos[1], path=väg)
else:
  print("❌ Ingen väg hittades!")

---
# 🔎 Del 5: Visualisera sökningen steg-för-steg

Nu ska vi se **exakt hur BFS utforskar världen** – steg för steg.

- `~` = Utforskad cell
- `*` = Den hittade vägen
- `R` = Robotens startposition

**Uppgift 5.1:** Kör cellen. Kan du se hur sökningen "sprider sig" utåt?

In [ ]:
# BFS med steg-för-steg visualisering
# =====================================

def bfs_visualisera(start, mål, world, max_steg=50):
  """BFS med utskrift av varje steg i sökningen"""

  kö        = deque([start])
  besökt    = {start}
  förälder  = {start: None}
  iteration = 0

  while kö and iteration < max_steg:
    nuvarande = kö.popleft()
    iteration += 1

    # Visa varje 5:e steg (annars för mycket utskrift)
    if iteration % 5 == 1 or nuvarande == mål:
      print(f"--- Iteration {iteration}: Utforskar {nuvarande} ---")
      print_grid(world,
                 robot_x=start[0], robot_y=start[1],
                 visited=list(besökt))

    if nuvarande == mål:
      # Rekonstruera vägen
      väg = []
      pos = nuvarande
      while pos is not None:
        väg.append(pos)
        pos = förälder[pos]
      väg = väg[::-1]
      print(f"🎯 MÅLET HITTAT efter {iteration} iterationer!")
      print(f"Kortaste väg ({len(väg)-1} steg): {' → '.join(str(p) for p in väg)}")
      print()
      print("Slutresultat - vägen markerad med *:")
      print_grid(world, robot_x=start[0], robot_y=start[1], path=väg)
      return väg

    for granne in get_neighbors(nuvarande, world):
      if granne not in besökt:
        besökt.add(granne)
        förälder[granne] = nuvarande
        kö.append(granne)

  print("❌ Ingen väg hittades")
  return []


# Kör visualiseringen!
print("BFS söker efter väg från", start_pos, "till", goal_pos)
print("=" * 50)
bfs_visualisera(start_pos, goal_pos, world)

---
# ✏️ Del 6: Praktiska övningar

Nu är det din tur att experimentera!

Lös övningarna nedan. Varje övning har `___` som du ska fylla i.

## Övning 5.1: Fyll i grannar

Förstå hur `get_neighbors` fungerar genom att fylla i de saknade delarna.

Hjälp: Koordinater i rutnätet:
```
         x ökar →
    y  (0,0) (1,0) (2,0)
    ↓  (0,1) (1,1) (2,1)
       (0,2) (1,2) (2,2)
```
- Höger → x ökar med 1
- Vänster → x minskar med 1
- Ner → y ökar med 1
- Upp → y minskar med 1

In [ ]:
# Övning 5.1: Fyll i de saknade delarna
# ----------------------------------------

def get_neighbors_övning(pos, world):
  x, y = pos
  neighbors = []

  # Höger: x ökar med 1
  if x + 1 < len(world[0]) and world[y][x+1] != CellType.WALL:
    neighbors.append((x + 1, y))

  # Vänster: x minskar med 1 - FYLL I NEDAN!
  if ___ < len(world[0]) and world[y][___] != CellType.WALL:
    neighbors.append((___, y))

  # Ner: y ökar med 1 - FYLL I NEDAN!
  if x < len(world[0]) and world[___][x] != CellType.WALL:
    neighbors.append((x, ___))

  # Upp: y minskar med 1 - FYLL I NEDAN!
  if x < len(world[0]) and world[___][x] != CellType.WALL:
    neighbors.append((x, ___))

  return neighbors


# Test - jämför med originalet
test_pos = (3, 3)
original = get_neighbors(test_pos, world)
# din_version = get_neighbors_övning(test_pos, world)
print("Original get_neighbors:", original)
# Avkommentera raden nedan när du fyllt i ovan:
# print("Din version:", din_version)
# print("Samma?", original == din_version)

## Övning 5.2: Hitta kortaste väg

Kör BFS mellan **valfria** positioner och se hur lång vägen blir.

**Uppgifter:**
1. Hitta vägen från (1,1) till (6,6)
2. Hitta vägen från (1,1) till (1,6)
3. Hitta vägen från (3,1) till (6,5)
4. Fråga: Är dessa de kortaste möjliga vägarna?

In [ ]:
# Övning 5.2: Prova olika start- och målpositioner
# --------------------------------------------------

# Ändra dessa värden och kör om cellen!
min_start = (1, 1)   # <-- Ändra start
mitt_mål  = (6, 6)   # <-- Ändra mål

väg = bfs_sök_väg(min_start, mitt_mål, world)

if väg:
  print(f"✅ Väg från {min_start} till {mitt_mål}:")
  print(f"   Antal steg: {len(väg) - 1}")
  print(f"   Vägen: {' → '.join(str(p) for p in väg)}")
  print()
  print_grid(world, robot_x=min_start[0], robot_y=min_start[1], path=väg)
else:
  print(f"❌ Ingen väg från {min_start} till {mitt_mål}")

## Övning 5.3: Jämför tre algoritmer

Nu jämför vi tre lösningar på samma problem:
1. **Slumpmässig** – väljer riktning slumpmässigt
2. **Regelbaserad** – följer enkla regler (från Lektion 4)
3. **BFS** – beräknar kortaste vägen

**Uppgift:** Kör cellen och se hur många steg varje algoritm tar.

Frågor:
- Vilken är snabbast?
- Vilken varierar mest?
- Varför är BFS alltid lika snabb?

In [ ]:
# Jämförelse: Slumpmässig vs Regelbaserad vs BFS
# ================================================

def slumpmässig_agent(start, mål, world, max_steg=1000):
  """Välj riktning slumpmässigt tills vi hittar målet"""
  x, y   = start
  steg   = 0
  riktningar = [(1,0), (-1,0), (0,1), (0,-1)]
  while (x, y) != mål and steg < max_steg:
    dx, dy = random.choice(riktningar)
    nx, ny = x + dx, y + dy
    if 0 <= nx < len(world[0]) and 0 <= ny < len(world):
      if world[ny][nx] != CellType.WALL:
        x, y = nx, ny
    steg += 1
  return steg if (x, y) == mål else None


def regelbaserad_agent(start, mål, world, max_steg=500):
  """Enkel regelbaserad: försök höger, ner, vänster, upp (i den ordningen)"""
  x, y  = start
  steg  = 0
  besökt = set()
  while (x, y) != mål and steg < max_steg:
    besökt.add((x, y))
    rörde_sig = False
    for dx, dy in [(1,0), (0,1), (-1,0), (0,-1)]:
      nx, ny = x + dx, y + dy
      if (0 <= nx < len(world[0]) and 0 <= ny < len(world)
          and world[ny][nx] != CellType.WALL
          and (nx, ny) not in besökt):
        x, y = nx, ny
        rörde_sig = True
        break
    if not rörde_sig:  # Fastnat! Börja om
      besökt.clear()
    steg += 1
  return steg if (x, y) == mål else None


# Kör alla tre och jämför!
print("Jämförelse: Slumpmässig vs Regelbaserad vs BFS")
print("=" * 50)

# BFS
väg_bfs = bfs_sök_väg(start_pos, goal_pos, world)
bfs_steg = len(väg_bfs) - 1 if väg_bfs else None

# Regelbaserad
regel_steg = regelbaserad_agent(start_pos, goal_pos, world)

# Slumpmässig (kör 5 gånger för att se variationen)
slump_steg_lista = [slumpmässig_agent(start_pos, goal_pos, world) for _ in range(5)]
slump_steg_snitt = sum(s for s in slump_steg_lista if s) / len(slump_steg_lista)

print(f"🧠 BFS:         {bfs_steg} steg (alltid samma!)")
print(f"📋 Regelbaserad: {regel_steg} steg")
print(f"🎲 Slumpmässig:  {slump_steg_lista} steg (varierar varje gång!)")
print(f"   Medelvärde slumpmässig: {slump_steg_snitt:.1f} steg")
print()
print("Världen med BFS-vägen:")
print_grid(world, robot_x=start_pos[0], robot_y=start_pos[1], path=väg_bfs)

## Övning 5.4: Städa ALLA smutsceller

Nu ska vi använda BFS för att städa **alla** smutsceller!

**Plan:**
1. Hitta alla smutsceller
2. Använd BFS för att planera vägen: Robot → Smuts 1 → Smuts 2 → ... → Hem
3. Kombinera alla delvägar till en stor plan

**Uppgift:** Kör koden och se hur roboten planerar sin resa!

In [ ]:
# Städa alla smutsceller med BFS
# =================================

# Skapa en ny värld med FLERA smutsceller
random.seed(7)
städ_värld = generate_world(8, wall_density=0.05, dirt_density=0.08)

# Hitta alla smutsceller i världen
def hitta_all_smuts(world):
  """Returnera lista med alla positioner som har smuts"""
  smuts_positioner = []
  for y in range(len(world)):
    for x in range(len(world[y])):
      if world[y][x] == CellType.DIRT:
        smuts_positioner.append((x, y))
  return smuts_positioner

start       = (1, 1)
all_smuts   = hitta_all_smuts(städ_värld)
hem         = (1, 1)

print(f"🏠 Start: {start}")
print(f"🦠 Smutsceller: {all_smuts}")
print(f"🏠 Hem: {hem}")
print()
print_grid(städ_värld, robot_x=start[0], robot_y=start[1])

# Planera vägen: start → smuts1 → smuts2 → ... → hem
total_väg = [start]
nuvarande = start

for smuts in all_smuts:
  del_väg = bfs_sök_väg(nuvarande, smuts, städ_värld)
  if del_väg:
    total_väg.extend(del_väg[1:])  # Lägg till (hoppa över start som redan finns)
    nuvarande = smuts
  else:
    print(f"⚠️ Kan inte nå {smuts}!")

# Väg tillbaka hem
väg_hem = bfs_sök_väg(nuvarande, hem, städ_värld)
if väg_hem:
  total_väg.extend(väg_hem[1:])

print(f"�� Total plan: {len(total_väg)-1} steg")
print(f"Väg: {' → '.join(str(p) for p in total_väg[:15])}..." if len(total_väg) > 15
      else f"Väg: {' → '.join(str(p) for p in total_väg)}")
print()
print("🗺️ Total resa (hela planen markerad med *):")
print_grid(städ_värld, robot_x=start[0], robot_y=start[1], path=total_väg)

## Övning 5.5: Hinderbana (Obstacle avoidance)

Vad händer om vägen är blockerad?

**Uppgifter:**
1. Kör koden med den nuvarande världen – hittar BFS en väg?
2. Lägg till en vägg på (3,1) och (3,2) och (3,3) – vad händer nu?
3. Blockera **hela** raden y=4 med väggar – vad händer?
4. Fråga: Kan BFS **alltid** fastna? Vad avgör det?

In [ ]:
# Övning 5.5: Hinderbana
# ========================

import copy

# Skapa en värld
random.seed(1)
hinder_värld = generate_world(8, wall_density=0.05, dirt_density=0.0)

# Lägg till en väggbarriär - ändra dessa för att experimentera!
# Ta bort # för att aktivera extra väggar:
# hinder_värld[1][3] = CellType.WALL  # Vägg på (3,1)
# hinder_värld[2][3] = CellType.WALL  # Vägg på (3,2)
# hinder_värld[3][3] = CellType.WALL  # Vägg på (3,3)

# Blockera hela rad 4 (kommentera in för att testa):
# for x in range(1, 7):
#   hinder_värld[4][x] = CellType.WALL

hinder_start = (1, 1)
hinder_mål   = (6, 6)

print("🗺️ Världen med hinder:")
print_grid(hinder_värld, robot_x=hinder_start[0], robot_y=hinder_start[1])

väg = bfs_sök_väg(hinder_start, hinder_mål, hinder_värld)

if väg:
  print(f"✅ Väg hittad! {len(väg)-1} steg")
  print_grid(hinder_värld, robot_x=hinder_start[0], robot_y=hinder_start[1], path=väg)
else:
  print("❌ Ingen väg hittades – målet är blockerat!")

---
# 🌳 Del 7: BONUS - DFT (Djupet-Först-Sökning)

BFS är inte den enda sökalgorithmen. En annan populär algoritm är **DFT (Depth-First Traversal)**.

## Skillnaden:

```
BFS (Bredden-Först):    DFT (Djupet-Först):
(1,1)                   (1,1)
├─(2,1)                 └─(2,1)
├─(1,2)                   └─(3,1)
├─(3,1)                     └─(4,1)
└─(2,2)                       └─(backa...)

BFS: Utforskar alla     DFT: Går så långt som
     grannar på samma        möjligt i en riktning
     nivå först               innan den backar
```

## Viktigt:
- **BFS:** Hittar alltid **kortaste** vägen ✅
- **DFT:** Besöker **alla** nåbara celler ✅, men ger **inte** kortaste vägen ❌

**BFS** använder en **KÖ** (popleft = FIFO)
**DFT** använder en **STACK** (pop = LIFO – sista in, första ut)

In [ ]:
# DFT - Djupet-Först-Sökning
# ============================
# Besöker ALLA nåbara celler från en startposition.
# Används för att traversera hela världen.

def dft_traversera(start, world):
  """
  Djupet-Först-Traversering från start.
  Returnerar alla nåbara positioner i besökningsordning.
  """
  stack   = [start]      # Stack – inte kö!
  besökt  = set()
  ordning = []

  while stack:
    nuvarande = stack.pop()   # Ta SISTA elementet (stack = LIFO)

    if nuvarande not in besökt:
      besökt.add(nuvarande)
      ordning.append(nuvarande)

      for granne in get_neighbors(nuvarande, world):
        if granne not in besökt:
          stack.append(granne)

  return ordning


# Testa DFT
random.seed(42)
test_värld = generate_world(8, wall_density=0.1, dirt_density=0.0)
start      = (1, 1)

dft_stig = dft_traversera(start, test_värld)
print(f"DFT besökte {len(dft_stig)} positioner")
print(f"Ordning (första 10): {dft_stig[:10]}")
print()
print("🗺️ DFT-stig (~ = besökt):")
print_grid(test_värld, robot_x=start[0], robot_y=start[1], visited=dft_stig)

### 🔄 Jämförelse: BFS vs DFT

**Uppgift:** Kör cellen nedan och jämför hur de två algoritmerna utforskar världen.

Frågor att fundera på:
- Besöker de samma positioner?
- Är det i samma ordning?
- Vilken verkar mer "logisk"?

In [ ]:
# Jämförelse: BFS vs DFT utforskningsordning
# ============================================

mål = (6, 6)

# BFS - kortaste vägen
bfs_väg = bfs_sök_väg(start, mål, test_värld)

# DFT - hela traverseringen
dft_stig = dft_traversera(start, test_värld)

print("BFS - Kortaste vägen till målet:")
print(f"  Antal steg: {len(bfs_väg)-1 if bfs_väg else 'n/a'}")
if bfs_väg:
  print_grid(test_värld, robot_x=start[0], robot_y=start[1], path=bfs_väg)

print("DFT - Hela traverseringen:")
print(f"  Besökte {len(dft_stig)} positioner")
print_grid(test_värld, robot_x=start[0], robot_y=start[1], visited=dft_stig)

print("Slutsats:")
print("  BFS: Hittar kortaste väg, men utforskar bara vad som behövs")
print("  DFT: Besöker hela världen, men ger inte kortaste vägen")

### DFT + BFS = Komplett städplan!

Vi kan kombinera algoritmerna:
1. Använd **DFT** för att hitta alla smutsceller (global traversering)
2. Använd **BFS** för att hitta kortaste vägen mellan dem

**Uppgift (BONUS):** Kör cellen och se den kompletta städplanen!

In [ ]:
# DFT + BFS: Komplett städplan
# =============================

random.seed(5)
komplett_värld = generate_world(8, wall_density=0.08, dirt_density=0.1)

# Steg 1: Hitta alla smutsceller med DFT (eller direkt sökning)
all_smuts = hitta_all_smuts(komplett_värld)
print(f"Hittade {len(all_smuts)} smutsceller: {all_smuts}")

# Steg 2: Planera vägen med BFS
start       = (1, 1)
nuvarande   = start
total_steg  = 0

for i, smuts in enumerate(all_smuts):
  del_väg = bfs_sök_väg(nuvarande, smuts, komplett_värld)
  if del_väg:
    print(f"  Smuts {i+1}: {nuvarande} → {smuts} = {len(del_väg)-1} steg")
    total_steg += len(del_väg) - 1
    nuvarande = smuts

# Väg hem
väg_hem = bfs_sök_väg(nuvarande, start, komplett_värld)
if väg_hem:
  print(f"  Hem: {nuvarande} → {start} = {len(väg_hem)-1} steg")
  total_steg += len(väg_hem) - 1

print(f"\n📊 Total plan: {total_steg} steg för att städa {len(all_smuts)} smutsceller")
print_grid(komplett_värld, robot_x=start[0], robot_y=start[1])

---
# 🧠 Del 8: Quiz

Testa din förståelse! Svara på frågorna nedan.

---

**Fråga 1:** Vad är en algoritm?

*Ditt svar:* ___

---

**Fråga 2:** Vad står BFS för?

*Ditt svar:* ___

---

**Fråga 3:** Varför är BFS bättre än en regelbaserad agent?

*Ditt svar:* ___

---

**Fråga 4:** Kan BFS fastna (hitta ingen väg)? Varför/varför inte?

*Ditt svar:* ___

---

**Fråga 5:** Vad är skillnaden mellan BFS och DFT?

*Ditt svar:* ___

---

**Fråga 6:** Vad är skillnaden mellan en **kö** (queue) och en **stack**?

*Ditt svar:* ___

## 📖 Quizsvar (Lärarversionersvarsversion)

*(Dölj denna cell för eleverna)*

**Svar 1:** En algoritm är en steg-för-steg-instruktion för att lösa ett problem.

**Svar 2:** BFS = Breadth-First Search = Bredden-Först-Sökning

**Svar 3:** BFS garanterar den kortaste vägen, regelbaserad kan fastna i loopar och är inte optimal.

**Svar 4:** Ja, om det inte finns någon väg (t.ex. målet är inneslutet av väggar). I det fallet returnerar BFS en tom lista.

**Svar 5:** BFS utforskar alla grannar på samma nivå (bredden-först, använder kö). DFT går så långt som möjligt i en riktning (djupet-först, använder stack). BFS hittar kortaste vägen, DFT besöker alla nåbara noder.

**Svar 6:** Kö (queue): Första in, första ut (FIFO) – som en kö på affären. Stack: Sista in, första ut (LIFO) – som en trave tallrikar.

---
# 🎉 Sammanfattning - Vad har vi lärt oss?

I den här lektionen har vi:

✅ **Förstått vad en algoritm är** – steg-för-steg instruktion

✅ **Jämfört tre sorters rörelse:**
   - 🎲 Slumpmässig (långsam, variabel)
   - 📋 Regelbaserad (kan fastna)
   - 🧠 BFS (optimal, alltid kortaste vägen)

✅ **Implementerat BFS** – algoritm som hittar kortaste vägen
   - Använder en **kö** (FIFO)
   - Håller koll på **besökta** positioner
   - Sparar **föräldralänkar** för att rekonstruera vägen

✅ **Visualiserat sökningen** – sett hur BFS sprider sig som ringar på vattnet

✅ **Städat hela världen optimalt** – BFS + DFT = komplett plan

✅ **Jämfört BFS och DFT:**
   - BFS = kortaste vägen (kö)
   - DFT = hela traverseringen (stack)

---

## 🚀 Nästa steg

Vad kan man göra härnäst?
- **A*-algoritmen** – ännu snabbare sökning med heuristik
- **Okänd miljö** – roboten utforskar världen medan den städar
- **Flera robotar** – koordinera flera städrobotar
- **Optimera ordningen** – vilket är bästa ordning att städa smutscellerna?